# PeakCut on Kaggle

Runs the full PeakCut server on Kaggle — transcription, ffmpeg rendering, AI clipping.
Your Mac just opens a browser tab.

**Before running:**
- Enable Internet in notebook settings
- Add Kaggle Secrets: `NGROK_AUTHTOKEN`, `PEAKCUT_AI_KEY`, `PEAKCUT_ASR_TOKEN`

## Cell 1 — Clone PeakCut

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Change this to your own fork if needed
REPO_URL = 'https://github.com/piotrswigon/PeakCut.git'

if not Path('PeakCut').exists():
    subprocess.run(['git', 'clone', REPO_URL, 'PeakCut'], check=True)
else:
    subprocess.run(['git', '-C', 'PeakCut', 'pull'], check=True)

os.chdir('PeakCut')
print('Ready:', os.getcwd())

## Cell 2 — Install requirements

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

import shutil
if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)
print('All done.')

## Cell 3 — Read secrets

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    _sc = UserSecretsClient()
    def get_secret(k):
        try: return _sc.get_secret(k) or ''
        except: return ''
except ImportError:
    def get_secret(k): return os.environ.get(k, '')

NGROK_TOKEN       = get_secret('NGROK_AUTHTOKEN')
PEAKCUT_AI_KEY    = get_secret('PEAKCUT_AI_KEY')
PEAKCUT_ASR_TOKEN = get_secret('PEAKCUT_ASR_TOKEN')

if not NGROK_TOKEN:
    raise RuntimeError('NGROK_AUTHTOKEN missing — add it in Add-ons > Secrets')

print('NGROK_AUTHTOKEN  :', 'OK' if NGROK_TOKEN else 'MISSING')
print('PEAKCUT_AI_KEY   :', 'OK' if PEAKCUT_AI_KEY else 'not set (clips by loudness only)')
print('PEAKCUT_ASR_TOKEN:', 'OK' if PEAKCUT_ASR_TOKEN else 'not set')

## Cell 4 — Download speech model (480 MB, one time)

In [ ]:
subprocess.run([sys.executable, '-m', 'studio.fetch_model'], check=True)

## Cell 5 — Start server + ngrok tunnel

Wait for the public URL to appear, then open it in your browser.

In [ ]:
import threading, time
from pyngrok import ngrok, conf

# Write .env
Path('.env').write_text(
    f'PEAKCUT_AI_PROVIDER=gemini\n'
    f'PEAKCUT_AI_KEY={PEAKCUT_AI_KEY}\n'
    f'PEAKCUT_ASR_TOKEN={PEAKCUT_ASR_TOKEN}\n',
    encoding='utf-8'
)

# Auth ngrok
conf.get_default().auth_token = NGROK_TOKEN

# Start PeakCut
env = os.environ.copy()
env['PEAKCUT_AI_KEY'] = PEAKCUT_AI_KEY
env['PEAKCUT_ASR_TOKEN'] = PEAKCUT_ASR_TOKEN

server = subprocess.Popen(
    [sys.executable, '-m', 'studio.server'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

def stream():
    for line in server.stdout:
        print(line, end='', flush=True)

threading.Thread(target=stream, daemon=True).start()
time.sleep(7)

# Open tunnel
tunnel = ngrok.connect(8765)
url = tunnel.public_url

print()
print('=' * 55)
print(f'  Open PeakCut at:  {url}')
print('=' * 55)
print('Keep this cell running.')

try:
    server.wait()
except KeyboardInterrupt:
    server.terminate()
    ngrok.disconnect(url)
    print('Stopped.')